# P0019 Experiment 03b: Enhanced Measurement
## Beyond SSC: Measuring Internal Coherence and Perspective-Aligned Structure

### Why this notebook exists

The original SSC measures "how well does the output follow the INPUT dependency structure (tau)."
This is the right metric for V0/V1, but for V2/V3 it penalizes exactly what the theory
predicts: reorganization away from tau toward a perspective-specific structure.

### New metrics

1. **ISC (Internal Semantic Coherence)**: Are semantically related concepts placed near each
   other in the output? Uses sentence-transformer embeddings of the output itself.

2. **Restructuring Index (RI)**: How much has the output reordered concepts relative to
   the input? High RI = heavy reorganization.

3. **Perspective Clustering (PC)**: Do concepts cluster according to perspective-relevant
   groupings rather than tau-based groupings?

4. **Output Semantic Density (OSD)**: How much semantic information is packed per sentence?
   Measured by mean pairwise similarity of consecutive sentence embeddings.

### Key insight driving the redesign

V1 maximizes SSC because "organize without perspective" = "follow tau."
V2/V3 SHOULD lower SSC because perspective-driven reorganization creates NEW structure.
The question is: can we measure that new structure?


In [ ]:
# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

!pip install -q sentence-transformers networkx scipy numpy pandas scikit-learn

import json, os, re
import numpy as np
import pandas as pd
import networkx as nx
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

BASE = '/content/drive/MyDrive/P0019'
OUT = f'{BASE}/exp03_measure'  # same dir, new files
os.makedirs(OUT, exist_ok=True)

# Load domains
with open(f'{BASE}/exp01_stimulus/exp01_domains.json') as f:
    DOMAINS = json.load(f)

# Load full responses
with open(f'{BASE}/exp02_collect/exp02_full_responses.json') as f:
    responses = json.load(f)
print(f"Loaded {len(responses)} responses")

# Load original SSC results for merging
orig_df = pd.read_csv(f'{OUT}/exp03_ssc_results.csv')
print(f"Original measurements: {len(orig_df)} rows")

In [ ]:
# === Load embedding model ===
# all-MiniLM-L6-v2: fast, good quality, 384-dim
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded: all-MiniLM-L6-v2 (384-dim)")

In [ ]:
# === Sentence splitter and concept extractor ===

def split_sentences(text):
    """Split text into sentences, filtering out very short ones."""
    # Remove markdown headers for cleaner splitting
    clean = re.sub(r"#{1,4}\s+[^\n]+", "", text)
    clean = re.sub(r"\*\*([^*]+)\*\*", r"\1", clean)  # remove bold
    sents = re.split(r"(?<=[.!?])\s+", clean.strip())
    return [s.strip() for s in sents if len(s.strip()) > 20]

def find_concept_mentions(text, concept_keys):
    """Find sentence index of each concept mention."""
    sentences = split_sentences(text)
    if not sentences:
        return {}, []
    positions = {}
    for key in concept_keys:
        patterns = [key.lower(), key.replace("_", " ").lower()]
        for si, sent in enumerate(sentences):
            for pat in patterns:
                if pat in sent.lower():
                    positions[key] = si
                    break
            if key in positions:
                break
        if key not in positions:
            positions[key] = len(sentences)
    return positions, sentences

# Test
test = "Temperature is important. Heat flows from hot to cold. The first law conserves energy."
pos, sents = find_concept_mentions(test, ["temperature", "heat", "first_law"])
print("Test:", pos, f"({len(sents)} sentences)")

## Metric 1: Internal Semantic Coherence (ISC)

ISC measures whether the output has organized concepts into a coherent semantic structure,
**regardless of whether that structure matches tau**.

Method:
1. Embed each sentence of the output using sentence-transformers
2. For each concept pair (i,j): measure their semantic distance in the OUTPUT
   = 1 - cosine_sim(sentence_embedding_i, sentence_embedding_j)
3. For each concept pair: measure their positional distance in the output
4. ISC = Spearman correlation between semantic distance and positional distance
   (high ISC = semantically similar concepts are placed near each other)


In [ ]:
# === ISC: Internal Semantic Coherence ===

def compute_isc(text, concept_keys, model):
    """Compute Internal Semantic Coherence.
    
    Measures whether the output places semantically related content nearby,
    regardless of original tau structure.
    """
    positions, sentences = find_concept_mentions(text, concept_keys)
    if len(sentences) < 3:
        return 0.0, 1.0, {}
    
    # Embed all sentences
    sent_embeddings = model.encode(sentences, show_progress_bar=False)
    
    # For each found concept, get its sentence embedding
    concept_embeds = {}
    for key, pos in positions.items():
        if pos < len(sentences):
            concept_embeds[key] = sent_embeddings[pos]
    
    found_keys = [k for k in concept_keys if k in concept_embeds]
    if len(found_keys) < 5:
        return 0.0, 1.0, {}
    
    # Pairwise semantic distances (in embedding space)
    sem_dists = []
    pos_dists = []
    for i, k1 in enumerate(found_keys):
        for j, k2 in enumerate(found_keys):
            if i < j:
                e1 = concept_embeds[k1].reshape(1, -1)
                e2 = concept_embeds[k2].reshape(1, -1)
                sem_d = 1.0 - cosine_similarity(e1, e2)[0, 0]
                pos_d = abs(positions[k1] - positions[k2])
                sem_dists.append(sem_d)
                pos_dists.append(pos_d)
    
    if len(sem_dists) < 10 or np.std(sem_dists) == 0 or np.std(pos_dists) == 0:
        return 0.0, 1.0, {}
    
    rho, p = stats.spearmanr(sem_dists, pos_dists)
    return rho, p, {"n_found": len(found_keys), "n_pairs": len(sem_dists)}

# Quick test
test_isc, test_p, test_info = compute_isc(
    "Temperature drives heat flow. Heat is energy transfer. "
    "Entropy measures disorder. The second law governs entropy increase. "
    "Work and internal energy relate through the first law. "
    "Carnot cycle achieves maximum efficiency for heat engines.",
    list(DOMAINS["S1"]["concepts"].keys()), model)
print(f"Test ISC: {test_isc:.3f} (p={test_p:.4f}), info={test_info}")

## Metric 2: Output Semantic Density (OSD)

Measures how tightly packed the semantic content is.
Higher V should produce denser, more integrated text.

Method: Mean cosine similarity between consecutive sentence embeddings.
High OSD = each sentence is semantically connected to the next (flowing argument).
Low OSD = disconnected listing.


In [ ]:
# === OSD: Output Semantic Density ===

def compute_osd(text, model):
    """Output Semantic Density: mean consecutive sentence similarity."""
    sentences = split_sentences(text)
    if len(sentences) < 3:
        return 0.0
    
    embeddings = model.encode(sentences, show_progress_bar=False)
    
    # Consecutive sentence similarities
    consec_sims = []
    for i in range(len(embeddings) - 1):
        e1 = embeddings[i].reshape(1, -1)
        e2 = embeddings[i + 1].reshape(1, -1)
        sim = cosine_similarity(e1, e2)[0, 0]
        consec_sims.append(sim)
    
    return float(np.mean(consec_sims))

## Metric 3: Restructuring Index (RI)

How much has the LLM reorganized the concepts relative to input order?

Method: Kendall tau between input order and output order.
RI = 1 - |kendall_tau|, so RI=1 means complete reorganization, RI=0 means same order.

V0 should have RI near 0 (faithful reproduction).
V2/V3 should have high RI (heavy reorganization).


In [ ]:
# === RI: Restructuring Index ===

def compute_ri(concept_order_input, positions_output, concept_keys):
    """Restructuring Index: how much the output reordered concepts."""
    # Input order: rank by position in concept_order_input
    input_ranks = {k: i for i, k in enumerate(concept_order_input)}
    
    # Output order: rank by position in output
    found = [(k, positions_output[k]) for k in concept_keys 
             if k in positions_output and positions_output[k] < 999]
    found.sort(key=lambda x: x[1])
    
    if len(found) < 5:
        return 0.0
    
    # Get input ranks for the found concepts in output order
    input_r = [input_ranks.get(k, 0) for k, _ in found]
    output_r = list(range(len(found)))
    
    tau, p = stats.kendalltau(input_r, output_r)
    
    # RI = 1 - |tau|: high RI = heavy restructuring
    return 1.0 - abs(tau)

## Metric 4: Perspective Divergence (PD)

For trials with specified psi: how different is this output from other psi directions
at the same V level, domain, and model?

This is computed in the analysis notebook (needs all trials together).
Here we just save the full-text embeddings for later comparison.


In [ ]:
# === Embed full outputs for Perspective Divergence ===

def compute_full_embedding(text, model):
    """Get a single embedding for the entire output text."""
    # Truncate if very long (model max ~256 tokens)
    if len(text) > 3000:
        text = text[:3000]
    return model.encode([text], show_progress_bar=False)[0]

In [ ]:
# === RUN ALL ENHANCED MEASUREMENTS ===

enhanced_results = []
full_embeddings = {}

for idx, resp in enumerate(responses):
    dk = resp["domain"]
    ckeys = list(DOMAINS[dk]["concepts"].keys())
    text = resp["response_text"]
    tid = resp["trial_id"]
    
    # Concept positions
    positions, sentences = find_concept_mentions(text, ckeys)
    
    # ISC
    isc, isc_p, isc_info = compute_isc(text, ckeys, model)
    
    # OSD
    osd = compute_osd(text, model)
    
    # RI
    ri = compute_ri(resp["concept_order"], positions, ckeys)
    
    # Full embedding (for perspective divergence)
    full_emb = compute_full_embedding(text, model)
    full_embeddings[tid] = full_emb
    
    enhanced_results.append({
        "trial_id": tid,
        "domain": dk,
        "v_level": resp["v_level"],
        "v_magnitude": resp["v_magnitude"],
        "psi_direction": resp["psi_direction"],
        "model": resp["model"],
        "repetition": resp["repetition"],
        "isc": round(float(isc), 4),
        "isc_p": round(float(isc_p), 6),
        "osd": round(float(osd), 4),
        "ri": round(float(ri), 4),
        "n_sentences": len(sentences),
    })
    
    if (idx + 1) % 50 == 0:
        print(f"  {idx+1}/{len(responses)}...")

print(f"\nEnhanced measurement complete: {len(enhanced_results)} trials")

In [ ]:
# === Compute Perspective Divergence (PD) ===
# For each trial with a psi direction, compute mean cosine distance
# to other psi directions at same (v_level, domain, model, rep)

for r in enhanced_results:
    r["pd"] = 0.0  # default

# Group by (v_level, domain, model, rep)
from collections import defaultdict
groups = defaultdict(list)
for r in enhanced_results:
    if r["psi_direction"] != "none":
        key = (r["v_level"], r["domain"], r["model"], r["repetition"])
        groups[key].append(r)

for key, trials in groups.items():
    for i, t1 in enumerate(trials):
        dists = []
        emb1 = full_embeddings[t1["trial_id"]].reshape(1, -1)
        for j, t2 in enumerate(trials):
            if i != j:
                emb2 = full_embeddings[t2["trial_id"]].reshape(1, -1)
                d = 1.0 - cosine_similarity(emb1, emb2)[0, 0]
                dists.append(d)
        if dists:
            t1["pd"] = round(float(np.mean(dists)), 4)

print("Perspective Divergence computed.")
# Quick check
for v in ["V2", "V3"]:
    sub = [r for r in enhanced_results if r["v_level"] == v]
    pd_vals = [r["pd"] for r in sub]
    print(f"  {v}: mean PD = {np.mean(pd_vals):.3f}")

In [ ]:
# === Merge with original SSC results and save ===

edf = pd.DataFrame(enhanced_results)

# Merge with original
merged = orig_df.merge(edf[["trial_id", "isc", "isc_p", "osd", "ri", "pd"]], 
                       on="trial_id", how="left")

# Save
merged.to_csv(f'{OUT}/exp03b_enhanced_results.csv', index=False)

# Save embeddings
np.savez_compressed(f'{OUT}/exp03b_embeddings.npz',
                    **{k: v for k, v in full_embeddings.items()})

print(f"Saved: {OUT}/exp03b_enhanced_results.csv ({len(merged)} rows)")
print(f"Saved: {OUT}/exp03b_embeddings.npz ({len(full_embeddings)} embeddings)")
print()
print("=== Enhanced Metrics Summary ===")
print(merged.groupby("v_level")[["ssc", "isc", "osd", "ri", "pd"]].mean().round(3))
print()
print("Proceed to P0019_exp04b_analyze.ipynb")